# Compile Workflows

Use Catalyst for Just-in-time (JIT) compilation

Need C++ simulators such as lightning.qubit

With autograph=True, it is possible to use Python logic (if / for)

In [ ]:
import pennylane as qml
import jax
import jax.numpy as jnp
import time


# Catalyst  does not support the pure-python 'default.qubit' -> lightning.qubit
dev_jit = qml.device("lightning.qubit", wires=2)

# @qml.qjit compiles the entire hybrid workflow into machine code
@qml.qjit(autograph=True) # use native Python 'if' statements inside
@qml.qnode(dev_jit, interface="jax")
def compiled_workflow(x):
    
    # Classical Control Flow 
    if x > 0.5:
        qml.Hadamard(wires=0)
    else:
        qml.PauliX(wires=0)
        
    # Quantum Operations
    qml.RY(x, wires=1)
    qml.CNOT(wires=[0, 1])
    
    return qml.expval(qml.PauliZ(1))

# JAX own numpy version (jnp) for high performance
param_1 = jnp.array(0.8)
param_2 = jnp.array(0.2)

# First run compiling
print("Compiling and executing 1st run...")
start_time_1 = time.perf_counter()
result_1 = compiled_workflow(param_1)
end_time_1 = time.perf_counter()
time_run_1 = end_time_1 - start_time_1

# Second run already compiled
print("Executing 2nd run...")
start_time_2 = time.perf_counter()
result_2 = compiled_workflow(param_2)
end_time_2 = time.perf_counter()
time_run_2 = end_time_2 - start_time_2

print(f"\n--- RESULTS ---")
print(f"Result 1 (x=0.8): {result_1:.4f} \t| Time: {time_run_1:.6f} seconds")
print(f"Result 2 (x=0.2): {result_2:.4f} \t| Time: {time_run_2:.6f} seconds")

Compiling and executing 1st run...
Executing 2nd run...

--- RESULTS ---
Result 1 (x=0.8): 0.0000 	| Time: 0.210709 seconds
Result 2 (x=0.2): -0.9801 	| Time: 0.000648 seconds


# Import Workflows

Can import circuits from other languages (Qiskit) in Pennylane, and calculate its gradients for using Machine Learning

In [2]:
import pennylane as qml
from qiskit import QuantumCircuit


# Circuit from Qiskit
qk_circuit = QuantumCircuit(2)
qk_circuit.h(0)
qk_circuit.cx(0, 1)
qk_circuit.rx(0.5, 1)

print("--- 1. ORIGINAL QISKIT CIRCUIT ---")
print(qk_circuit.draw())


# Import into PennyLane
pl_qfunc = qml.from_qiskit(qk_circuit)


# Embed it into a PennyLane QNode
dev_import = qml.device("default.qubit", wires=2)

@qml.qnode(dev_import)
def hybrid_imported_circuit():
    
    # Call the imported function and map it
    pl_qfunc(wires=[0, 1])
    
    # Add Pennylane gates
    qml.PauliZ(wires=0)
    
    # Define PennyLane measurements
    return qml.expval(qml.PauliZ(1))

print("\n--- 2. CONVERTED PENNYLANE CIRCUIT ---")
print(qml.draw(hybrid_imported_circuit)())

print(f"\n--- 3. EXECUTION RESULT ---")
print(f"Expectation Value: {hybrid_imported_circuit():.4f}")

--- 1. ORIGINAL QISKIT CIRCUIT ---
     ┌───┐                
q_0: ┤ H ├──■─────────────
     └───┘┌─┴─┐┌─────────┐
q_1: ─────┤ X ├┤ Rx(0.5) ├
          └───┘└─────────┘

--- 2. CONVERTED PENNYLANE CIRCUIT ---
0: ──H─╭●──Z────────┤     
1: ────╰X──RX(0.50)─┤  <Z>

--- 3. EXECUTION RESULT ---
Expectation Value: 0.0000
